# Robust Haskell DPO Pipeline v2


In [1]:
%pip install -q "openai>=1.0.0" "datasets>=2.0.0" "python-dotenv>=1.0.0" "tqdm>=4.0.0"


In [2]:
import shutil
if shutil.which('runghc') is None:
    !apt-get update -qq
    !apt-get install -y -qq ghc
else:
    print('runghc:', shutil.which('runghc'))


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not handshake: Error in the pull function. [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out [IP: 185.125.190.80 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.
Selecting previously unselected package libgmpxx4ldbl:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../libgmpxx4ldbl_2%3a6.2.1+dfsg-3ubuntu1_amd64.deb ...
Unpacking libgmpxx4ldbl:amd64 (2:6.2.1+dfsg-3ubuntu1) ...
Selecting previously unselected package libgmp-dev:amd64.
Preparing to u

In [ ]:
import os
from pathlib import Path

# PASTE IN
# RESOURCE_GROUP  =  
# OPENAI_API_KEY  =
# OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"

API_VERSION = "2025-04-01-preview"

DPO_MODEL = 'gpt-4.1-nano-2025-04-14'
CANDIDATE_MODEL = 'gpt-4.1-mini'
DATASET_NAME = 'finbarr/rlvr-code-data-haskell-edited'

SUBSAMPLE_FRACTION = 0.10

USE_RAW_JSONL_SPLITS = True
RAW_TRAIN_FILE = Path('train_raw.jsonl')
RAW_VAL_FILE = Path('val_raw.jsonl')
PAIR_ROW_LIMIT = 200
ALLOW_VAL_FALLBACK_FROM_TRAIN_RAW = True
RANDOM_SEED = 42
TRAIN_VAL_TEST_SPLIT = (0.85, 0.10, 0.05)

DPO_N_CANDIDATES = 8
DPO_TEMPERATURE = 0.4
DPO_TOP_P = 0.90
GEN_CONCURRENCY = 8
DPO_MIN_SCORE_GAP = 1

# For debugging, set these to 30. For full run, set to None.
MAX_TRAIN_ROWS_FOR_PAIR_BUILDING = None
MAX_VAL_ROWS_FOR_PAIR_BUILDING = None

DPO_N_EPOCHS = 1
DPO_BATCH_SIZE = 8
DPO_LEARNING_RATE_MULT = 0.05

INFERENCE_MAX_TOKENS = 512
INFERENCE_TEMPERATURE = 0.0
INFERENCE_CONCURRENCY = 16

OUT_DIR = Path('dpo_outputs')
OUT_DIR.mkdir(exist_ok=True)
DPO_TRAIN_FILE = OUT_DIR / 'dpo_training.jsonl'
DPO_VAL_FILE = OUT_DIR / 'dpo_validation.jsonl'
TEST_FILE = OUT_DIR / 'test.jsonl'
PAIR_DEBUG_FILE = OUT_DIR / 'pair_debug.jsonl'
PREDICTIONS_OUT = OUT_DIR / 'predictions.jsonl'

SYSTEM_PROMPT = (
    'You are an expert Haskell programmer. '
    'Given a natural-language description of a programming task, '
    'write a correct, idiomatic Haskell function implementation. '
    'Output only the Haskell code — no markdown fences, no preamble, no explanation.'
)

print('endpoint:', OPENAI_ENDPOINT)
print('api key loaded:', bool(OPENAI_API_KEY and not OPENAI_API_KEY.startswith('<')))


endpoint: https://cis-5270-team-9.openai.azure.com
api key loaded: True


In [4]:
import asyncio, json, random, re, subprocess, tempfile
from collections import Counter
from pathlib import Path
from typing import Any, Optional
from datasets import load_dataset
from openai import AzureOpenAI, AsyncAzureOpenAI, RateLimitError, APIError
from tqdm.auto import tqdm

def write_jsonl(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')

def count_jsonl(path):
    if not path.exists():
        return 0
    with open(path, encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

def assert_nonempty_jsonl(path):
    n = count_jsonl(path)
    if n == 0:
        raise ValueError(f'{path} is empty. Do not launch DPO.')
    print(f'{path}: {n:,} rows')

openai_client = AzureOpenAI(api_key=OPENAI_API_KEY, azure_endpoint=OPENAI_ENDPOINT, api_version='2025-04-01-preview')
async_openai_client = AsyncAzureOpenAI(api_key=OPENAI_API_KEY, azure_endpoint=OPENAI_ENDPOINT, api_version='2025-04-01-preview')
print('clients initialized')


clients initialized


In [6]:
def load_jsonl(path: Path) -> list[dict]:
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

if USE_RAW_JSONL_SPLITS:
    if not RAW_TRAIN_FILE.exists():
        raise FileNotFoundError(f'Missing RAW_TRAIN_FILE: {RAW_TRAIN_FILE}')

    train_raw = load_jsonl(RAW_TRAIN_FILE)
    print(f'loaded train_raw: {len(train_raw):,} rows from {RAW_TRAIN_FILE}')

    train_data = train_raw[:PAIR_ROW_LIMIT]

    if RAW_VAL_FILE.exists():
        val_raw = load_jsonl(RAW_VAL_FILE)
        print(f'loaded val_raw: {len(val_raw):,} rows from {RAW_VAL_FILE}')
        val_data = val_raw[:PAIR_ROW_LIMIT]
    elif ALLOW_VAL_FALLBACK_FROM_TRAIN_RAW:
        val_data = train_raw[PAIR_ROW_LIMIT:2 * PAIR_ROW_LIMIT]
        print(
            f'WARNING: {RAW_VAL_FILE} not found. '
            f'Using train_raw rows {PAIR_ROW_LIMIT}:{2 * PAIR_ROW_LIMIT} as val_data.'
        )
    else:
        raise FileNotFoundError(f'Missing RAW_VAL_FILE: {RAW_VAL_FILE}')

    test_data = train_raw[2 * PAIR_ROW_LIMIT:3 * PAIR_ROW_LIMIT]
    write_jsonl(test_data, TEST_FILE)

    print('train_data for pair building:', len(train_data))
    print('val_data for pair building:', len(val_data))
    print('test_data saved:', len(test_data), '->', TEST_FILE)
    print('keys:', list(train_data[0].keys()))
else:
    ds = load_dataset(DATASET_NAME, split='train')
    data = list(ds)
    rng = random.Random(RANDOM_SEED)
    rng.shuffle(data)
    n = max(20, int(len(data) * SUBSAMPLE_FRACTION))
    data = data[:n]
    f_train, f_val, f_test = TRAIN_VAL_TEST_SPLIT
    i_tr = int(n * f_train)
    i_va = int(n * (f_train + f_val))
    train_data = data[:i_tr]
    val_data = data[i_tr:i_va]
    test_data = data[i_va:]
    write_jsonl(test_data, TEST_FILE)
    print('full dataset:', len(ds))
    print('subsample:', n)
    print('train/val/test:', len(train_data), len(val_data), len(test_data))
    print('keys:', list(train_data[0].keys()))


loaded train_raw: 1,600 rows from train_raw.jsonl
train_data for pair building: 200
val_data for pair building: 200
test_data saved: 200 -> dpo_outputs/test.jsonl
keys: ['translated_problem', 'translated_solution', 'translated_test_cases', 'id', 'messages', 'ground_truth', 'target_language']


In [7]:
def clean_haskell_output(text: str) -> str:
    text = (text or '').strip()
    text = re.sub(r'^```(?:haskell|hs)?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)
    text = re.sub(r'^\s*module\s+[A-Za-z0-9_.]+\s*(?:\([^)]*\))?\s*where\s*', '', text, flags=re.MULTILINE)
    lines = text.splitlines()
    code_start = re.compile(r"^\s*(import\s+|{-#|[a-zA-Z_][\w']*\s*(::|=)|data\s+|type\s+|newtype\s+|class\s+|instance\s+)")
    while lines and not code_start.search(lines[0]):
        lines.pop(0)
    return '\n'.join(lines).strip()

def normalize_solution_module(solution_code: str) -> str:
    return 'module Solution where\n\n' + clean_haskell_output(solution_code or '').strip() + '\n'

def first_present(row, keys):
    for k in keys:
        if k in row and row[k] not in (None, ''):
            return row[k]
    return None

def get_problem(row):
    return str(first_present(row, ['translated_problem','problem','prompt','question','description','instruction']) or '')

def get_reference_solution(row):
    v = first_present(row, ['translated_solution','solution','canonical_solution','completion','answer','code'])
    return clean_haskell_output(str(v)) if v is not None else None

def get_tests(row):
    v = first_present(row, ['translated_test_cases','test_cases','tests','unit_tests','test'])
    if v is None:
        return []
    if isinstance(v, list):
        return [str(x) for x in v]
    if isinstance(v, str):
        s = v.strip()
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
        except Exception:
            pass
        return [s]
    return [str(v)]

def inspect_row(row):
    print('KEYS:', list(row.keys()))
    print('\nPROBLEM:\n', get_problem(row)[:1200])
    print('\nREFERENCE:\n', (get_reference_solution(row) or '')[:1200])
    print('\nTESTS:')
    for t in get_tests(row)[:3]:
        print(repr(t[:1200]))


In [8]:
MAIN_IMPORTS = [
    'import Solution',
    'import qualified Data.Map.Strict as Map',
    'import qualified Data.Map as MapLazy',
    'import qualified Data.Set as Set',
    'import qualified Data.List as List',
    'import qualified Data.Sequence as Seq',
    'import qualified Data.IntMap.Strict as IntMap',
    'import qualified Data.IntSet as IntSet',
    'import Data.Char', 'import Data.Maybe', 'import Data.Either',
    'import Data.Ord', 'import Data.Function',
    'import Control.Monad', 'import Control.Applicative',
    'import Control.Exception (evaluate, SomeException, try)',
]

def classify_tests(tests):
    if not tests:
        return 'expression'
    joined = '\n'.join(map(str, tests))
    if re.search(r'^\s*module\s+Main\s+where', joined, re.MULTILINE):
        return 'full_program'
    if any(re.search(r'^\s*import\s+', str(t), re.MULTILINE) or re.search(r'^\s*main\s*(::|=)', str(t), re.MULTILINE) for t in tests):
        return 'full_program'
    if all(str(t).strip().startswith('--') or str(t).strip() == '' for t in tests):
        return 'commented'
    return 'expression'

def build_expression_main(tests):
    defs, checks = [], []
    for i, test in enumerate(tests):
        defs += [f'test_{i} :: Bool', f'test_{i} = ({str(test).strip()})', '']
        checks.append(f'  putStrLn $ if test_{i} then "PASS_{i}" else "FAIL_{i}"')
    return '\n'.join(['module Main where', *MAIN_IMPORTS, '', *defs, 'main :: IO ()', 'main = do', *checks, ''])

def build_full_program_main(tests):
    test = '\n'.join(map(str, tests)).strip()
    if re.search(r'^\s*module\s+Main\s+where', test, flags=re.MULTILINE):
        return test + '\n'
    test_body = re.sub(r'^\s*module\s+\w+\s*(?:\([^)]*\))?\s*where\s*\n?', '', test, flags=re.MULTILINE)
    return '\n'.join(['module Main where', *MAIN_IMPORTS, '', test_body, ''])

def build_compile_only_main():
    return '\n'.join(['module Main where', *MAIN_IMPORTS, '', 'main :: IO ()', 'main = putStrLn "PASS_0"', ''])

def infer_total(stdout, tests, fmt):
    lines = [line.strip() for line in stdout.splitlines() if line.strip()]
    if fmt == 'full_program' and lines and all(line in {'True','False'} for line in lines):
        return len(lines)
    if fmt == 'commented':
        return 1
    return len(tests)

def score_stdout(stdout, total, fmt):
    lines = [line.strip() for line in stdout.splitlines() if line.strip()]
    pass_markers = sum(1 for line in lines if line.startswith('PASS_'))
    if pass_markers:
        return pass_markers
    if lines and all(line in {'True','False'} for line in lines):
        return sum(1 for line in lines if line == 'True')
    if fmt == 'commented':
        return total
    return 0

def run_tests_debug(solution_code, tests, show_files=False):
    fmt = classify_tests(tests)
    with tempfile.TemporaryDirectory() as tmpdir:
        sol_text = normalize_solution_module(solution_code)
        if fmt == 'expression':
            main_code = build_expression_main(tests)
        elif fmt == 'full_program':
            main_code = build_full_program_main(tests)
        else:
            main_code = build_compile_only_main()
        sol_path = Path(tmpdir) / 'Solution.hs'
        main_path = Path(tmpdir) / 'Main.hs'
        sol_path.write_text(sol_text, encoding='utf-8')
        main_path.write_text(main_code, encoding='utf-8')
        try:
            result = subprocess.run(['runghc', '-i' + tmpdir, str(main_path)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=20)
        except subprocess.TimeoutExpired:
            out = {'compiled': False, 'passed': 0, 'total': len(tests), 'fmt': fmt, 'stdout': '', 'stderr': 'TimeoutExpired'}
            if show_files: out.update({'Solution.hs': sol_text, 'Main.hs': main_code})
            return out
        except FileNotFoundError:
            out = {'compiled': False, 'passed': 0, 'total': len(tests), 'fmt': fmt, 'stdout': '', 'stderr': 'runghc not found. Install GHC first.'}
            if show_files: out.update({'Solution.hs': sol_text, 'Main.hs': main_code})
            return out
        if result.returncode != 0:
            out = {'compiled': False, 'passed': 0, 'total': len(tests), 'fmt': fmt, 'stdout': result.stdout[-4000:], 'stderr': result.stderr[-8000:]}
        else:
            total = infer_total(result.stdout, tests, fmt)
            out = {'compiled': True, 'passed': score_stdout(result.stdout, total, fmt), 'total': total, 'fmt': fmt, 'stdout': result.stdout[-4000:], 'stderr': result.stderr[-8000:]}
        if show_files:
            out.update({'Solution.hs': sol_text, 'Main.hs': main_code})
        return out

def run_tests(solution_code, tests):
    return run_tests_debug(solution_code, tests, show_files=False)


In [9]:
sanity = run_tests_debug('add :: Int -> Int -> Int\nadd x y = x + y', ['add 2 3 == 5', 'add 1 1 == 2'], show_files=True)
print('expression sanity:', {k: sanity[k] for k in ['compiled','passed','total','fmt']})
assert sanity['compiled'] and sanity['passed'] == sanity['total'], sanity

solution = '''
import qualified Data.Set as Set

findIntersection :: (Ord a) => [a] -> [a] -> [a]
findIntersection list1 list2 = Set.toList $ Set.fromList list1 `Set.intersection` Set.fromList list2
'''
full_program_test = '''
module Main where
import Solution
import qualified Data.Set as Set

main :: IO ()
main = do
    print $ Set.fromList (findIntersection [1,2,3,4] [3,4,5,6]) == Set.fromList [3,4]
    print $ Set.fromList (findIntersection [] [1,2,3]) == Set.empty
    print $ Set.fromList (findIntersection ['a','b','c'] ['b','c','d']) == Set.fromList ['b','c']
'''
sanity2 = run_tests_debug(solution, [full_program_test], show_files=True)
print('full-program sanity:', {k: sanity2[k] for k in ['compiled','passed','total','fmt']})
print('stdout:', sanity2['stdout'])
if not sanity2['compiled']:
    print(sanity2['stderr'])
assert sanity2['compiled'] and sanity2['passed'] == sanity2['total'], sanity2


expression sanity: {'compiled': True, 'passed': 2, 'total': 2, 'fmt': 'expression'}
full-program sanity: {'compiled': True, 'passed': 3, 'total': 3, 'fmt': 'full_program'}
stdout: True
True
True



## Prompt and candidate generation

In [11]:
def make_training_prompt(row, max_visible_tests=8):
    problem = get_problem(row)
    visible_tests = '\n'.join(f'-- {t}' for t in get_tests(row)[:max_visible_tests])
    return (
        'Write Haskell code that will be placed directly inside this file:\n\n'
        'module Solution where\n\n'
        f'Task:\n{problem}\n\n'
        f'The code must satisfy tests like:\n\n{visible_tests}\n\n'
        'Rules:\n'
        '- Output only Haskell code.\n'
        '- Do not include markdown fences.\n'
        '- Do not include explanations.\n'
        '- Do not include "module Solution where".\n'
        '- Define all functions needed by the tests.\n'
        '- Prefer simple total functions.\n'
    )

async def chat_completion_with_retries(client, messages, *, model, temperature, top_p, max_tokens, attempts=4):
    last_err = None
    for attempt in range(attempts):
        try:
            resp = await client.chat.completions.create(model=model, messages=messages, temperature=temperature, top_p=top_p, max_tokens=max_tokens)
            return resp.choices[0].message.content or ''
        except (RateLimitError, APIError) as e:
            last_err = e
            await asyncio.sleep(min(30, 2 ** attempt))
        except Exception as e:
            last_err = e
            await asyncio.sleep(min(30, 2 ** attempt))
    raise RuntimeError(f'chat completion failed after {attempts} attempts: {last_err}')

async def generate_candidates_async(client, prompt):
    async def one():
        raw = await chat_completion_with_retries(
            client,
            messages=[{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':prompt}],
            model=CANDIDATE_MODEL,
            temperature=DPO_TEMPERATURE,
            top_p=DPO_TOP_P,
            max_tokens=INFERENCE_MAX_TOKENS,
        )
        return clean_haskell_output(raw)
    outputs = await asyncio.gather(*[one() for _ in range(DPO_N_CANDIDATES)], return_exceptions=True)
    cleaned = []
    for out in outputs:
        if isinstance(out, Exception):
            continue
        out = clean_haskell_output(out)
        if out.strip():
            cleaned.append(out)
    seen, unique = set(), []
    for c in cleaned:
        if c not in seen:
            unique.append(c); seen.add(c)
    return unique

print(make_training_prompt(train_data[0])[:2000])


Write Haskell code that will be placed directly inside this file:

module Solution where

Task:
Write a Haskell function with the signature fibonacci :: Int -> [Int] that returns a list containing the first n numbers of the Fibonacci sequence, using recursion. For example, fibonacci 5 should return [0, 1, 1, 2, 3].

The code must satisfy tests like:

-- fibonacci 1 == [0]
-- fibonacci 2 == [0,1]
-- fibonacci 3 == [0,1,1]
-- fibonacci 5 == [0,1,1,2,3]
-- fibonacci 7 == [0,1,1,2,3,5,8]
-- fibonacci 0 == []
-- fibonacci 10 == [0,1,1,2,3,5,8,13,21,34]

Rules:
- Output only Haskell code.
- Do not include markdown fences.
- Do not include explanations.
- Do not include "module Solution where".
- Define all functions needed by the tests.
- Prefer simple total functions.



## Build DPO preference pairs

In [13]:
def make_dpo_row(prompt: str, preferred: str, rejected: str, meta: dict) -> dict:
    return {
        "input": {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ]
        },
        "preferred_output": [
            {"role": "assistant", "content": clean_haskell_output(preferred)}
        ],
        "non_preferred_output": [
            {"role": "assistant", "content": clean_haskell_output(rejected)}
        ],
        "_meta": meta,
    }


def strip_meta(row: dict) -> dict:
    row = dict(row)
    row.pop("_meta", None)
    return row


async def build_pairs_for_one_row(client, row: dict, row_i: int):
    debug = []

    problem = get_problem(row)
    reference = get_reference_solution(row)
    tests = get_tests(row)

    if not problem:
        debug.append({"row_i": row_i, "event": "skip", "reason": "missing_problem"})
        return [], debug

    if not tests:
        debug.append({"row_i": row_i, "event": "skip", "reason": "missing_tests"})
        return [], debug

    prompt = make_training_prompt(row)

    ref_score = None
    if reference:
        ref_score = run_tests(reference, tests)
        debug.append({
            "row_i": row_i,
            "event": "reference_score",
            "compiled": ref_score["compiled"],
            "passed": ref_score["passed"],
            "total": ref_score["total"],
            "fmt": ref_score["fmt"],
        })
    else:
        debug.append({"row_i": row_i, "event": "missing_reference"})

    candidates = await generate_candidates_async(client, prompt)
    if not candidates:
        debug.append({"row_i": row_i, "event": "skip", "reason": "no_candidates"})
        return [], debug

    scored = []
    for cand in candidates:
        score = run_tests(cand, tests)
        scored.append((cand, score))

    scored.sort(key=lambda x: (x[1]["passed"], int(x[1]["compiled"])))

    worst_code, worst_score = scored[0]
    best_code, best_score = scored[-1]

    debug.append({
        "row_i": row_i,
        "event": "model_scores",
        "num_candidates": len(scored),
        "best_compiled": best_score["compiled"],
        "best_passed": best_score["passed"],
        "best_total": best_score["total"],
        "worst_compiled": worst_score["compiled"],
        "worst_passed": worst_score["passed"],
        "worst_total": worst_score["total"],
        "fmt": best_score["fmt"],
    })

    pairs = []

    # On-policy pair: best model candidate vs worst model candidate.
    if best_score["passed"] - worst_score["passed"] >= DPO_MIN_SCORE_GAP:
        pairs.append(make_dpo_row(
            prompt,
            best_code,
            worst_code,
            {
                "kind": "on_policy_best_vs_worst",
                "row_i": row_i,
                "best_passed": best_score["passed"],
                "worst_passed": worst_score["passed"],
                "total": best_score["total"],
                "best_compiled": best_score["compiled"],
                "worst_compiled": worst_score["compiled"],
            },
        ))

    # Bootstrap pair: reference solution vs worst model candidate.
    if (
        reference
        and ref_score
        and ref_score["compiled"]
        and ref_score["passed"] - worst_score["passed"] >= DPO_MIN_SCORE_GAP
    ):
        pairs.append(make_dpo_row(
            prompt,
            reference,
            worst_code,
            {
                "kind": "bootstrap_reference_vs_model",
                "row_i": row_i,
                "best_passed": ref_score["passed"],
                "worst_passed": worst_score["passed"],
                "total": ref_score["total"],
                "best_compiled": ref_score["compiled"],
                "worst_compiled": worst_score["compiled"],
            },
        ))

    if not pairs:
        debug.append({
            "row_i": row_i,
            "event": "skip",
            "reason": "no_positive_score_gap",
            "ref_passed": ref_score["passed"] if ref_score else None,
            "best_model_passed": best_score["passed"],
            "worst_model_passed": worst_score["passed"],
        })

    return pairs, debug


async def build_pairs_dataset(
    client,
    rows: list[dict],
    *,
    split_name: str,
    max_rows=None,
    concurrency=None,
):
    if max_rows is not None:
        rows = rows[:max_rows]

    if concurrency is None:
        concurrency = GEN_CONCURRENCY

    sem = asyncio.Semaphore(concurrency)
    all_pairs = []
    all_debug = []

    async def worker(i_row):
        i, row = i_row
        async with sem:
            try:
                return await build_pairs_for_one_row(client, row, i)
            except Exception as e:
                return [], [{"row_i": i, "event": "exception", "reason": repr(e)}]

    tasks = [worker(x) for x in enumerate(rows)]

    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f"build {split_name} pairs"):
        pairs, debug = await fut
        all_pairs.extend(pairs)
        all_debug.extend(debug)

    kind_counts = Counter(p["_meta"]["kind"] for p in all_pairs)
    skip_reasons = Counter(
        d.get("reason", "unknown")
        for d in all_debug
        if d.get("event") in {"skip", "exception"}
    )

    best_passed = [p["_meta"]["best_passed"] for p in all_pairs]
    worst_passed = [p["_meta"]["worst_passed"] for p in all_pairs]

    summary = {
        "split": split_name,
        "rows_seen": len(rows),
        "kept_pairs": len(all_pairs),
        "kind_counts": dict(kind_counts),
        "skip_reasons": dict(skip_reasons),
        "avg_best_passed": sum(best_passed) / len(best_passed) if best_passed else 0.0,
        "avg_worst_passed": sum(worst_passed) / len(worst_passed) if worst_passed else 0.0,
    }

    print(json.dumps(summary, indent=2))
    return all_pairs, all_debug, summary

In [14]:
train_pairs, train_debug, train_pair_summary = await build_pairs_dataset(
    async_openai_client,
    train_data,
    split_name='train_raw_first_200',
    max_rows=None,  # train_data is already sliced to PAIR_ROW_LIMIT
)

val_pairs, val_debug, val_pair_summary = await build_pairs_dataset(
    async_openai_client,
    val_data,
    split_name='val_raw_first_200',
    max_rows=None,  # val_data is already sliced to PAIR_ROW_LIMIT
)


build train_raw_first_200 pairs:   0%|          | 0/200 [00:00<?, ?it/s]

{
  "split": "train_raw_first_200",
  "rows_seen": 200,
  "kept_pairs": 76,
  "kind_counts": {
    "on_policy_best_vs_worst": 44,
    "bootstrap_reference_vs_model": 32
  },
  "skip_reasons": {
    "no_positive_score_gap": 152,
    "no_candidates": 1
  },
  "avg_best_passed": 8.644736842105264,
  "avg_worst_passed": 1.7236842105263157
}


build val_raw_first_200 pairs:   0%|          | 0/200 [00:00<?, ?it/s]

{
  "split": "val_raw_first_200",
  "rows_seen": 200,
  "kept_pairs": 65,
  "kind_counts": {
    "on_policy_best_vs_worst": 34,
    "bootstrap_reference_vs_model": 31
  },
  "skip_reasons": {
    "no_positive_score_gap": 161
  },
  "avg_best_passed": 10.184615384615384,
  "avg_worst_passed": 0.6461538461538462
}


## Save and inspect DPO files

In [15]:
debug_rows = [{'split':'train', **d} for d in train_debug] + [{'split':'val', **d} for d in val_debug]
write_jsonl(debug_rows, PAIR_DEBUG_FILE)
write_jsonl([strip_meta(p) for p in train_pairs], DPO_TRAIN_FILE)
write_jsonl([strip_meta(p) for p in val_pairs], DPO_VAL_FILE)
print('train rows:', count_jsonl(DPO_TRAIN_FILE))
print('val rows:', count_jsonl(DPO_VAL_FILE))
print('debug rows:', count_jsonl(PAIR_DEBUG_FILE))
assert_nonempty_jsonl(DPO_TRAIN_FILE)
assert_nonempty_jsonl(DPO_VAL_FILE)

def inspect_pair(pair):
    print('META:', json.dumps(pair.get('_meta', {}), indent=2))
    print('\nPROMPT:\n', pair['input']['messages'][-1]['content'][:1500])
    print('\nPREFERRED:\n', pair['preferred_output'][0]['content'][:1500])
    print('\nREJECTED:\n', pair['non_preferred_output'][0]['content'][:1500])

inspect_pair(train_pairs[0])


train rows: 76
val rows: 65
debug rows: 1113
dpo_outputs/dpo_training.jsonl: 76 rows
dpo_outputs/dpo_validation.jsonl: 65 rows
META: {
  "kind": "on_policy_best_vs_worst",
  "row_i": 39,
  "best_passed": 12,
  "worst_passed": 0,
  "total": 12,
  "best_compiled": true,
  "worst_compiled": false
}

PROMPT:
 Write Haskell code that will be placed directly inside this file:

module Solution where

Task:
One day Jeff got hold of an integer sequence a1, a2, ..., an of length n. The boy immediately decided to analyze the sequence. For that, he needs to find all values of x, for which these conditions hold:

  * x occurs in sequence a.
  * Consider all positions of numbers x in the sequence a (such i, that ai = x). These numbers, sorted in increasing order, must form an arithmetic progression. 

Help Jeff, find all x that meet the problem conditions.

Write a function with the following signature:

analyzeArithmeticPositions :: Int -> [Int] -> [(Int, Int)]

The function should return a list of

Launch dpo job


In [19]:
import time
from pathlib import Path

DPO_MODEL = "gpt-4.1-mini-2025-04-14"

DPO_N_EPOCHS = 1
DPO_BATCH_SIZE = 1
DPO_LEARNING_RATE_MULT = 0.05

DPO_TRAIN_FILE = Path("dpo_outputs/dpo_training.jsonl")
DPO_VAL_FILE = Path("dpo_outputs/dpo_validation.jsonl")


def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def assert_nonempty_jsonl(path: Path) -> None:
    n = count_jsonl(path)
    if n == 0:
        raise ValueError(f"{path} is empty or missing.")
    print(f"{path}: {n:,} rows")


def wait_for_file_import(file_id: str, timeout_s: int = 900, poll_s: int = 10):
    """
    Azure/OpenAI file uploads may return before the file is fully imported.
    Fine-tuning requires the file status to be completed/processed first.
    """
    start = time.time()

    while True:
        f = openai_client.files.retrieve(file_id)
        status = getattr(f, "status", None)

        # Defensive raw dump support across client versions.
        raw = f.model_dump() if hasattr(f, "model_dump") else {}
        raw_status = raw.get("status")
        status = status or raw_status

        print(f"{file_id}: status={status}")

        if status in {"processed", "completed", "succeeded"}:
            print(f"{file_id}: import complete")
            return f

        if status in {"failed", "error", "cancelled"}:
            raise RuntimeError(f"File import failed for {file_id}: {raw or f}")

        if time.time() - start > timeout_s:
            raise TimeoutError(f"Timed out waiting for file import: {file_id}")

        time.sleep(poll_s)

#validation

assert_nonempty_jsonl(DPO_TRAIN_FILE)
assert_nonempty_jsonl(DPO_VAL_FILE)

print("\nUsing:")
print("DPO_MODEL:", DPO_MODEL)
print("DPO_TRAIN_FILE:", DPO_TRAIN_FILE)
print("DPO_VAL_FILE:", DPO_VAL_FILE)



print("\nUploading training file...")
dpo_train_upload = openai_client.files.create(
    file=open(DPO_TRAIN_FILE, "rb"),
    purpose="fine-tune",
)

print("Uploading validation file...")
dpo_val_upload = openai_client.files.create(
    file=open(DPO_VAL_FILE, "rb"),
    purpose="fine-tune",
)

dpo_train_id = dpo_train_upload.id
dpo_val_id = dpo_val_upload.id

print("\nUploaded:")
print("train:", dpo_train_id)
print("val:  ", dpo_val_id)


# wait on file import due to lag of azure upload

print("\nWaiting for training file import...")
wait_for_file_import(dpo_train_id)

print("\nWaiting for validation file import...")
wait_for_file_import(dpo_val_id)


# create DPO
print("\nCreating DPO fine-tuning job...")

dpo_job = openai_client.fine_tuning.jobs.create(
    model=DPO_MODEL,
    training_file=dpo_train_id,
    validation_file=dpo_val_id,
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {
                "n_epochs": DPO_N_EPOCHS,
                "batch_size": DPO_BATCH_SIZE,
                "learning_rate_multiplier": DPO_LEARNING_RATE_MULT,
            }
        },
    },
    extra_body={"trainingType": "GlobalStandard"},
    suffix="haskell-dpo",
)

print("\nDPO job created.")
print("Job ID :", dpo_job.id)
print("Status :", dpo_job.status)
print("Model  :", dpo_job.model)

DPO_JOB_ID = dpo_job.id

dpo_outputs/dpo_training.jsonl: 76 rows
dpo_outputs/dpo_validation.jsonl: 65 rows

Using:
DPO_MODEL: gpt-4.1-mini-2025-04-14
DPO_TRAIN_FILE: dpo_outputs/dpo_training.jsonl
DPO_VAL_FILE: dpo_outputs/dpo_validation.jsonl

Uploading training file...
Uploading validation file...

Uploaded:
train: file-e96671a3284b456a9c66f4dd5687165c
val:   file-82aebcb17bb843f5bf96c034bddc4d15

Waiting for training file import...
file-e96671a3284b456a9c66f4dd5687165c: status=pending
file-e96671a3284b456a9c66f4dd5687165c: status=processed
file-e96671a3284b456a9c66f4dd5687165c: import complete

Waiting for validation file import...
file-82aebcb17bb843f5bf96c034bddc4d15: status=processed
file-82aebcb17bb843f5bf96c034bddc4d15: import complete

Creating DPO fine-tuning job...

DPO job created.
Job ID : ftjob-c9e3433154a248ab953ec38090cb9621
Status : pending
Model  : gpt-4.1-mini-2025-04-14


## Monitor job

In [ ]:
JOB_ID = None
if JOB_ID:
    print(openai_client.fine_tuning.jobs.retrieve(JOB_ID))
else:
    print('Set JOB_ID to monitor a launched job.')


In [ ]:
FINE_TUNED_DEPLOYMENT = None

async def predict_one(client, deployment, row):
    raw = await chat_completion_with_retries(
        client,
        messages=[{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':make_training_prompt(row)}],
        model=deployment,
        temperature=INFERENCE_TEMPERATURE,
        top_p=1.0,
        max_tokens=INFERENCE_MAX_TOKENS,
    )
    pred = clean_haskell_output(raw)
    return {'problem': get_problem(row), 'prediction': pred, 'score': run_tests_debug(pred, get_tests(row))}

async def evaluate_deployment(client, deployment, rows, max_rows=None, concurrency=INFERENCE_CONCURRENCY):
    if max_rows is not None:
        rows = rows[:max_rows]
    sem = asyncio.Semaphore(concurrency)
    async def worker(row):
        async with sem:
            try:
                return await predict_one(client, deployment, row)
            except Exception as e:
                return {'problem': get_problem(row), 'prediction':'', 'score': {'compiled':False, 'passed':0, 'total':len(get_tests(row)), 'fmt':'exception', 'stdout':'', 'stderr':repr(e)}}
    tasks = [worker(row) for row in rows]
    results = []
    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc='evaluate'):
        results.append(await fut)
    return results

if FINE_TUNED_DEPLOYMENT:
    eval_results = await evaluate_deployment(async_openai_client, FINE_TUNED_DEPLOYMENT, test_data)
    write_jsonl(eval_results, PREDICTIONS_OUT)
    compiled = sum(r['score']['compiled'] for r in eval_results)
    passed = sum(r['score']['passed'] for r in eval_results)
    total = sum(r['score']['total'] for r in eval_results)
    print('compiled:', compiled, '/', len(eval_results))
    print('tests passed:', passed, '/', total, f'= {passed / max(total, 1):.2%}')
else:
    print('Set FINE_TUNED_DEPLOYMENT after deploying the fine-tuned model.')
